# Skycart quickstart

A tour of Skycart against your own Raiser's Edge NXT database.

Before running this you need:
1. `pip install "skycart[pandas] @ git+https://github.com/ngoodroe/skycart.git"`
2. A `.env` file in this folder (or any folder above it) with your Blackbaud app credentials. See `.env.example` and the README.

## 1. Log in (one time)

This opens your browser to Blackbaud's approval screen. After you approve, Skycart saves your login to `~/.skycart/tokens.json` and refreshes it automatically from then on, so you can skip this cell next time.

Running Jupyter inside Docker or on a remote server? Use `skycart.login(paste=True)` instead.

In [ ]:
import skycart

skycart.login()

## 2. Get one constituent

`skycart.get()` takes any SKY API path and returns plain Python data (dicts and lists). Swap in a real constituent system record ID from your database.

In [ ]:
CONSTITUENT_ID = "280"  # <- change me

constituent = skycart.get_constituent(CONSTITUENT_ID)
constituent

## 3. Prospect status, with days in status

This is the data that's hard to get out of RE queries: how long someone has been sitting in their current prospect status.

In [ ]:
skycart.get_prospect_status(CONSTITUENT_ID)

## 4. A whole portfolio as a table

Pull one fundraiser's assignments, look up each prospect's status, and turn it all into a pandas DataFrame. `get_prospect_statuses()` makes one API call per constituent, so start small.

In [ ]:
FUNDRAISER_ID = "123"  # <- a gift officer's constituent system record ID

assignments = skycart.get_fundraiser_assignments(FUNDRAISER_ID)
prospect_ids = [a["constituent_id"] for a in assignments]

statuses = skycart.to_dataframe(skycart.get_prospect_statuses(prospect_ids))
statuses.sort_values("days_elapsed", ascending=False)

## 5. Any list endpoint, every page

`skycart.get_all()` follows Blackbaud's paging for you. `limit` keeps it from pulling your entire database while you're exploring.

In [ ]:
recent = skycart.get_all("/constituent/v1/constituents", limit=50)
skycart.to_dataframe(recent).head()

## 6. Any endpoint at all

Find an endpoint in the [SKY API reference](https://developer.blackbaud.com/skyapi/products/renxt) and pass its path to `skycart.get()`. Errors come back as `skycart.SkyAPIError`, with Blackbaud's own message.

In [ ]:
skycart.get(f"/constituent/v1/constituents/{CONSTITUENT_ID}/relationships")